This project is created using OLLAMA version

In [4]:
%pip install -U ddgs  

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from ddgs import DDGS

topic = input("Enter Topic: ")

all_content = []

with DDGS() as ddgs:

    search_results = ddgs.text(
        topic,
        max_results=10
    )

    

In [22]:
for result in search_results:

    # Print once to see the structure (remove later)
    print(result)

    # Works for both ddgs.text() and ddgs.news()
    url = result.get("url") or result.get("href")

    if not url:
        print("No URL found. Skipping...")
        continue

    print(f"\nExtracting: {url}")

    try:
        # Extract the full article
        article = ddgs.extract(url, fmt="text_markdown")

        # Handle different return formats
        if isinstance(article, dict):
            content = article.get("content", "")

        elif isinstance(article, list):
            if len(article) > 0:
                first = article[0]
                if isinstance(first, dict):
                    content = first.get("content", "")
                else:
                    content = str(first)
            else:
                content = ""

        else:
            content = str(article)

        # Fallback if extract returned empty
        if not content.strip():
            content = result.get("body", "")

        if content.strip():
            all_content.append(content)
            print(content[:300])
        else:
            print("No content available.")

    except Exception as e:
        print("Extraction Error:", e)

        # Fallback to the snippet returned by DDGS
        body = result.get("body", "")

        if body:
            print("Using news snippet instead...")
            all_content.append(body)
            print(body[:300])

{'title': 'Global warming', 'href': 'https://en.wikipedia.org/wiki/Global_warming', 'body': 'Present-day climate change includes both global warming—the ongoing increase in global average temperature—and its wider effects on Earth\'s climate system. Climate change in a broader sense also includes previous long-term changes to Earth\'s climate. The modern-day rise in global temperatures is driven by human activities, especially fossil fuel (coal, oil and natural gas) burning since the Industrial Revolution. Fossil fuel use, deforestation, and some agricultural and industrial practices release greenhouse gases. These gases absorb some of the heat that the Earth radiates after it warms from sunlight, warming the lower atmosphere. Earth\'s atmosphere now has roughly 50% more carbon dioxide, the main gas driving global warming, than it did at the end of the pre-industrial era, reaching levels not seen for millions of years.Climate change has an increasingly large impact on the environment. 

In [23]:
import os
import json
from langchain_ollama import ChatOllama


client = ChatOllama(
    model="gemma3:4b",      # or gemma:2b if that's the model you downloaded
    temperature=0.3,
)



# Storing summaries for each source
Final_summaries = []

# Loop through each content and generate summary
for idx, content in enumerate(all_content):
    # from the above command it means idx is the index of the content in the all_content list and content is the actual content extracted from the URL.

    print(f"\nProcessing Source {idx+1}")
# the above 
    # Prevent huge inputs
    content = content[:5000]

    prompt = f"""
You are an expert research analyst.

Analyze the following content and return ONLY valid JSON.
Figure out all the entities in the article,place and date to specifically mention in the summary.
Make sure to not repeat the statement to point out the information . 

Required JSON Structure:

{{
    "summary": "150 word summary of the content",
}}

Topic:
{topic}

Content:
{content}
"""

    try:

        response = client.invoke(prompt)

        cleaned_response = response.content.strip()

        # Remove markdown if present
        cleaned_response = cleaned_response.replace("```json", "")
        cleaned_response = cleaned_response.replace("```", "")
        cleaned_response = cleaned_response.strip()

        parsed_json = json.loads(cleaned_response)

        Final_summaries.append({
            "source_id": idx + 1,
            "parsed_content": parsed_json
        })

        print(f"Source {idx+1} Processed Successfully")

    except Exception as e:

        print(f"Error in Source {idx+1}: {e}")

# =====================================
# Save JSON
# =====================================
with open(
    "source_summaries.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        Final_summaries,
        f,
        # FILE OBJECTS ARE SAVED IN A HUMAN-READABLE FORMAT
        indent=4,
        ensure_ascii=False
    )

print("\nSaved Successfully!")
print("File: source_summaries.json")




Processing Source 1
Source 1 Processed Successfully

Processing Source 2
Source 2 Processed Successfully

Processing Source 3
Source 3 Processed Successfully

Processing Source 4
Source 4 Processed Successfully

Processing Source 5
Source 5 Processed Successfully

Processing Source 6
Source 6 Processed Successfully

Processing Source 7
Source 7 Processed Successfully

Processing Source 8
Source 8 Processed Successfully

Processing Source 9
Source 9 Processed Successfully

Processing Source 10
Source 10 Processed Successfully

Processing Source 11
Source 11 Processed Successfully

Processing Source 12
Source 12 Processed Successfully

Processing Source 13
Source 13 Processed Successfully

Processing Source 14
Source 14 Processed Successfully

Processing Source 15
Source 15 Processed Successfully

Processing Source 16
Source 16 Processed Successfully

Processing Source 17
Source 17 Processed Successfully

Processing Source 18
Source 18 Processed Successfully

Processing Source 19
Source

In [24]:
import os
import json


client = ChatOllama(
    model="gemma3:4b",      # or gemma:2b if that's the model you downloaded
    temperature=0.3,
)

# Read the summaries from the JSON file
with open("source_summaries.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)


combined_summary = ""

for source in summaries:

    summary = source["parsed_content"]["summary"]

    combined_summary += summary + "\n\n"

print("="*80)
print("TOTAL SUMMARY LENGTH :", len(combined_summary))
print("="*80)

prompt = f"""
You are a professional journalist and senior news editor.

Your task is to convert the provided summaries into a single, accurate, publication-ready news article.

### Instructions

* Carefully read and understand all the summaries.
* Identify the main event and merge similar information without repetition.
* Preserve all important facts, including names, organizations, locations, dates, statistics, and timelines.
* Do not add, assume, exaggerate, or omit any factual information.
* Maintain a formal, objective, and neutral journalistic tone.

Important Instructions:
- Generate only the final news article.
- Do NOT ask the user any questions.
- Do NOT request additional information or clarification.
- Do NOT include suggestions, recommendations, notes, disclaimers, or conversational text.
- Do NOT include phrases such as "To help me refine this analysis...", "Please provide...", "Could you tell me...","Let me know..." or "would u like....".
- Do NOT include quotes unless they are explicitly present in the provided summary.
- Do NOT invent or attribute statements or quotations to any person or organization.
- The output must begin directly with the news headline and end with the conclusion.
- Return only the article. No additional text before or after the article.

* Ensure the article clearly answers:

  * What happened?
  * Where did it happen?
  * When did it happen?
  * Who was involved?
  * Why is it important?
* Write approximately **500 words**.

### Output Format

# Headline

Write a clear and engaging headline.

# Introduction

Write one introductory paragraph summarizing the main event.

# Article

Write **exactly three well-developed paragraphs**. Add **subheadings only where they naturally fit**. Each paragraph should cover:

1. Background and context
2. Key developments
3. Impact and significance

# Conclusion

Write a short concluding paragraph highlighting the overall significance of the event.


===========================
SUMMARY TO ANALYZE
===========================

{combined_summary}

"""


print("\nGenerating Final Article...\n")

response = client.invoke(prompt)

article = response.content

with open(
    "final_article.json",
    "w",
    encoding="utf-8"
) as f:

    f.write(article)

print("Final Article Saved Successfully!")
print("File : final_article.md")

print("\n")
print(article)

TOTAL SUMMARY LENGTH : 12201

Generating Final Article...

Final Article Saved Successfully!
File : final_article.md


# Global Warming: A Crisis of Human-Induced Change

# Introduction

The scientific community has reached a near-universal consensus: Earth’s climate is changing at an unprecedented rate, primarily driven by human activities.  As of August 10, 2025, the planet has experienced its warmest year on record, with a surface temperature increase of +1.60°C (2.88°F) above pre-industrial levels – a trend solidified through decades of research and data analysis. This “global warming,” as it’s often termed, represents a significant challenge demanding immediate mitigation strategies to avert potentially catastrophic consequences for ecosystems and human populations worldwide.

# Key Developments & Scientific Understanding 

The core issue revolves around the greenhouse effect, intensified by rising concentrations of atmospheric gases like carbon dioxide (CO2) and methane. Since th